CrossGatedHyena / PureGeometricHyena — Band Gap Prediction

## 0 · Configuration — fill in before running

In [ ]:
# ── Repo ──────────────────────────────────────────────────────────────────
GITHUB_REPO   = "https://github.com/ShreyPatel1311/Cross-Gated-Hyena.git"
REPO_NAME     = "/Files"
# If private repo:  https://TOKEN@github.com/USER/REPO.git

# ── Data ──────────────────────────────────────────────────────────────────
HF_REPO_ID    = "Godseye1311/alignn-band-gap"
DATA_DIR      = "/content/Files"
CKPT_DIR      = "/content/Files"
CKPT_NAME     = "best_model.pt"

# ── Model selector ────────────────────────────────────────────────────────
# "crossgated"  → model.py         (CrossGatedHyena, Hyena + cross-gating)
# "pure_hyena"  → model_pure_hyena.py  (PureGeometricHyena, no cross-gating)
MODEL         = "crossgated"

# ── Model hyperparameters ─────────────────────────────────────────────────
NODE_DIM      = 128
EDGE_DIM      = 128    # used by crossgated only
NUM_LAYERS    = 4
NUM_RBF_POS   = 64
NUM_RBF_ANGLE = 32
MAX_K         = 16
FILTER_HIDDEN = 128
DROPOUT       = 0.1

# ── Training ──────────────────────────────────────────────────────────────
EPOCHS        = 10
EDGE_BUDGET   = 25_000
LR            = 1e-3
WEIGHT_DECAY  = 1e-4
NUM_WORKERS   = 4
MAX_IDS       = None      # None = full ~100K; set e.g. 2000 for quick test
ACCUM_STEPS   = 4
RESUME        = False
AUGMENT       = True      # inversion-symmetry augmentation during training


## 1 · Clone repo

In [ ]:
import os, sys

REPO_DIR = f"/content/{REPO_NAME}"

if os.path.exists(REPO_DIR):
    print(f"Repo already at {REPO_DIR} — pulling latest …")
    os.system(f"git -C {REPO_DIR} pull")
else:
    print(f"Cloning {GITHUB_REPO} …")
    ret = os.system(f"git clone {GITHUB_REPO} {REPO_DIR}")
    if ret != 0:
        raise RuntimeError("git clone failed — check GITHUB_REPO and token")

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

for fname in ["model.py", "model_attn.py", "model_pure_hyena.py", "dataset.py", "graph_utils.py"]:
    path = os.path.join(REPO_DIR, fname)
    print(f"  {'✓' if os.path.exists(path) else '✗ MISSING'}  {fname}")


In [3]:
import time, os, torch.nn as nn
import torch
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from tqdm.notebook import tqdm
import subprocess, sys
from huggingface_hub import hf_hub_download

print(f"PyTorch {torch.__version__}  |  CUDA {torch.version.cuda}")

# PyTorch Geometric — must match torch+cuda version
pyg_url = (
    f"https://data.pyg.org/whl/torch-"
    f"{torch.__version__.split('+')[0]}+cu{torch.version.cuda.replace('.','')}.html"
)
subprocess.check_call([sys.executable, "-m", "pip", "install",
    "torch_geometric", "torch_scatter", "torch_sparse",
    "-f", pyg_url, "-q"])

subprocess.check_call([sys.executable, "-m", "pip",
    "install", "h5py", "huggingface_hub", "tqdm", "-q"])

print("All packages installed.")

PyTorch 2.9.0+cu128  |  CUDA 12.8


error: externally-managed-environment

× This environment is externally managed
╰─> To install Python packages system-wide, try apt install
    python3-xyz, where xyz is the package you are trying to
    install.
    
    If you wish to install a non-Debian-packaged Python package,
    create a virtual environment using python3 -m venv path/to/venv.
    Then use path/to/venv/bin/python and path/to/venv/bin/pip. Make
    sure you have python3-full installed.
    
    If you wish to install a non-Debian packaged Python application,
    it may be easiest to use pipx install xyz, which will manage a
    virtual environment for you. Make sure you have pipx installed.
    
    See /usr/share/doc/python3.12/README.venv for more information.

note: If you believe this is a mistake, please contact your Python installation or OS distribution provider. You can override this, at the risk of breaking your Python installation or OS, by passing --break-system-packages.
hint: See PEP 668 for the detai

CalledProcessError: Command '['/usr/bin/python3', '-m', 'pip', 'install', 'torch_geometric', 'torch_scatter', 'torch_sparse', '-f', 'https://data.pyg.org/whl/torch-2.9.0+cu128.html', '-q']' returned non-zero exit status 1.

## 3 · GPU + AMP dtype

In [13]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if device.type == "cuda":
    p   = torch.cuda.get_device_properties(0)
    cap = (p.major, p.minor)
    print(f"GPU  : {p.name}  (compute {cap[0]}.{cap[1]})")
    print(f"VRAM : {p.total_memory / 1e9:.1f} GB")

    # BF16 Tensor Cores: Ampere (8.0+). T4 = 7.5 → FP16 + GradScaler
    AMP_DTYPE       = torch.bfloat16 if cap[0] >= 8 else torch.float16
    USE_GRAD_SCALER = (AMP_DTYPE == torch.float16)
    USE_AMP         = True

    # TF32: free ~10% throughput on matmuls (Ampere+)
    torch.backends.cuda.matmul.fp32_precision = "tf32"
    torch.backends.cudnn.conv.fp32_precision  = "tf32"
    print(f"AMP  : {AMP_DTYPE}  |  GradScaler : {USE_GRAD_SCALER}")
    print("TF32 : enabled")
else:
    AMP_DTYPE = torch.float32; USE_AMP = False; USE_GRAD_SCALER = False
    print("WARNING: No GPU — training will be very slow.")

print(f"Device : {device}")

GPU  : NVIDIA T500  (compute 7.5)
VRAM : 3.9 GB
AMP  : torch.float16  |  GradScaler : True
TF32 : enabled
Device : cuda


## 4 · Download data from HuggingFace

> Upload your files first if you haven't already:
> ```bash
> huggingface-cli upload YOUR_USERNAME/band-gap-prediction graphs_data.h5 --repo-type dataset
> huggingface-cli upload YOUR_USERNAME/band-gap-prediction materials_tabular.csv --repo-type dataset
> ```


In [10]:
os.makedirs(DATA_DIR,  exist_ok=True)
os.makedirs(CKPT_DIR,  exist_ok=True)

H5_PATH  = os.path.join(DATA_DIR, "graphs_data.h5")
CSV_PATH = os.path.join(DATA_DIR, "materials_tabular.csv")

for fname, local in [("graphs_data.h5", H5_PATH), ("materials_tabular.csv", CSV_PATH)]:
    if os.path.exists(local):
        print(f"{fname}: already present ({os.path.getsize(local)/1e9:.2f} GB)")
        continue
    print(f"Downloading {fname} …")
    hf_hub_download(repo_id=HF_REPO_ID, filename=fname, repo_type="dataset", local_dir=DATA_DIR)
    print(f"{fname}: done")
print("\nData ready.")

graphs_data.h5: already present (9.43 GB)
materials_tabular.csv: already present (0.01 GB)

Data ready.


In [6]:
# Search entire /content for .h5 files
!find /content -name "*.h5" 2>/dev/null

# Search for the CSV too
!find /content -name "*.csv" 2>/dev/null

/content/graphs_data.h5
/content/materials_tabular.csv
/content/sample_data/california_housing_test.csv
/content/sample_data/mnist_train_small.csv
/content/sample_data/mnist_test.csv
/content/sample_data/california_housing_train.csv


## 5 · Import from cloned repo

In [ ]:
from dataset import CrossGatedHyenaDataset, EdgeBudgetBatchSampler
from torch_geometric.loader import DataLoader
from torch.utils.data import random_split
import torch

if MODEL == "crossgated":
    from model import CrossGatedHyena as Model
else:
    from model_pure_hyena import PureGeometricHyena as Model

print(f"Model  : {Model.__name__}  (from {Model.__module__})")
print(f"Dataset: CrossGatedHyenaDataset  |  target: band_gap (eV)")


## 6 · Dataset + `EdgeBudgetBatchSampler`

- `CrossGatedHyenaDataset`: 3.4× faster than ALIGNNDataset — no line-graph build
- `EdgeBudgetBatchSampler`: each batch ≤ `EDGE_BUDGET` total edges  
  → no OOM from variable-size graphs; small crystals batch together efficiently


In [ ]:
print("Indexing dataset …")
stats   = torch.load(os.path.join(DATA_DIR, "node_stats.pt"))
full_ds = CrossGatedHyenaDataset(
    H5_PATH, CSV_PATH,
    num_rbf=NUM_RBF_POS, max_ids=MAX_IDS,
    node_stats=stats, augment=AUGMENT,
)
print(f"Total  : {len(full_ds):,} materials")

n = len(full_ds)
n_tr = int(0.80*n);  n_val = int(0.10*n);  n_te = n - n_tr - n_val
gen  = torch.Generator().manual_seed(42)
tr_ds, val_ds, te_ds = random_split(full_ds, [n_tr, n_val, n_te], generator=gen)
print(f"Train {n_tr:,}  Val {n_val:,}  Test {n_te:,}")

tr_samp  = EdgeBudgetBatchSampler(tr_ds,  EDGE_BUDGET, shuffle=True)
val_samp = EdgeBudgetBatchSampler(val_ds, EDGE_BUDGET, shuffle=False)
te_samp  = EdgeBudgetBatchSampler(te_ds,  EDGE_BUDGET, shuffle=False)
print(f"Train batches : {len(tr_samp):,}  (EDGE_BUDGET={EDGE_BUDGET:,})")

_ldr = dict(num_workers=NUM_WORKERS, pin_memory=True,
            persistent_workers=(NUM_WORKERS > 0),
            prefetch_factor=2 if NUM_WORKERS > 0 else None)
train_loader = DataLoader(tr_ds,  batch_sampler=tr_samp,  **_ldr)
val_loader   = DataLoader(val_ds, batch_sampler=val_samp, **_ldr)
test_loader  = DataLoader(te_ds,  batch_sampler=te_samp,  **_ldr)

b = next(iter(train_loader))
print(f"\nSample batch:")
print(f"  x_cat     : {b.x_cat.shape}    x : {b.x.shape}")
print(f"  edge_cat  : {b.edge_cat.shape}  edge_attr : {b.edge_attr.shape}")
print(f"  y         : {b.y.shape}  — band_gap (eV)")
print(f"  band_gap  : [{b.y.min():.2f}, {b.y.max():.2f}] eV")


## 7 · Model

In [ ]:
if MODEL == "crossgated":
    model = Model(
        node_in_dim            = 8,
        edge_in_dim            = NUM_RBF_POS,
        node_dim               = NODE_DIM,
        edge_dim               = EDGE_DIM,
        num_layers             = NUM_LAYERS,
        num_rbf_pos            = NUM_RBF_POS,
        num_rbf_angle          = NUM_RBF_ANGLE,
        max_k                  = MAX_K,
        filter_hidden          = FILTER_HIDDEN,
        dropout                = DROPOUT,
        cutoff                 = 6.0,
        gradient_checkpointing = True,
    ).to(device)
else:
    model = Model(
        node_in_dim            = 8,
        node_dim               = NODE_DIM,
        num_layers             = NUM_LAYERS,
        num_rbf_pos            = NUM_RBF_POS,
        num_rbf_angle          = NUM_RBF_ANGLE,
        max_k                  = MAX_K,
        filter_hidden          = FILTER_HIDDEN,
        dropout                = DROPOUT,
        cutoff                 = 6.0,
        gradient_checkpointing = True,
    ).to(device)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"{Model.__name__} — {n_params:,} parameters")
print(f"  max_k={MAX_K}  |  layers={NUM_LAYERS}  |  dim={NODE_DIM}  |  AMP={AMP_DTYPE}")

model.eval()
with torch.no_grad():
    with torch.autocast(device.type, dtype=AMP_DTYPE, enabled=USE_AMP):
        out = model(b.to(device))
print(f"\nForward check: {tuple(out.shape)}  ✓  — band gap (eV)")


## 8 · Training

In [ ]:
criterion = nn.HuberLoss(delta=0.5)   # band gap in log1p space
l1_loss   = nn.L1Loss()

optim     = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = CosineAnnealingLR(optim, T_max=EPOCHS, eta_min=LR * 1e-3)
scaler    = torch.amp.GradScaler(enabled=USE_GRAD_SCALER, init_scale=2**14, growth_interval=5000)

ckpt_path    = os.path.join(CKPT_DIR, CKPT_NAME)
best_val_mae = float("inf")
start_epoch  = 1
history = {"loss": [], "train_mae": [], "val_mae": [], "val_rmse": []}

if RESUME and os.path.exists(ckpt_path):
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    model.load_state_dict(ckpt["model"])
    optim.load_state_dict(ckpt["optim"])
    scheduler.load_state_dict(ckpt["scheduler"])
    scaler.load_state_dict(ckpt["scaler"])
    best_val_mae = ckpt["val_mae"]
    start_epoch  = ckpt["epoch"] + 1
    history      = ckpt.get("history", history)
    print(f"Resumed from epoch {ckpt['epoch']}  (val MAE = {best_val_mae:.4f} eV)")
else:
    print(f"Fresh training — epochs 1..{EPOCHS}")


@torch.no_grad()
def evaluate(loader, desc="eval"):
    model.eval()
    mae = mse = n = 0
    for b in tqdm(loader, desc=desc, leave=False):
        b = b.to(device, non_blocking=True)
        with torch.autocast(device.type, dtype=AMP_DTYPE, enabled=USE_AMP):
            pred = model(b).float().squeeze(-1)           # (B,)
        tgt  = b.y.float().squeeze(-1)                   # (B,)
        pred_eV = torch.expm1(pred)                      # undo log1p
        mae += (pred_eV - tgt).abs().sum().item()
        mse += ((pred_eV - tgt) ** 2).sum().item()
        n   += tgt.numel()
    return mae / n, (mse / n) ** 0.5


for epoch in range(start_epoch, EPOCHS + 1):
    tr_samp.set_epoch(epoch)
    full_ds.train()

    model.train()
    t0 = time.perf_counter()
    total_loss = total_mae = n_step = 0
    optim.zero_grad(set_to_none=True)

    bar = tqdm(train_loader, desc=f"Ep {epoch:3d}/{EPOCHS}", leave=True)
    for step, b in enumerate(bar):
        b = b.to(device, non_blocking=True)

        with torch.autocast(device.type, dtype=AMP_DTYPE, enabled=USE_AMP):
            pred = model(b).squeeze(-1)                             # (B,)
            tgt  = torch.log1p(b.y.float().squeeze(-1).clamp(min=0))  # log1p(band_gap)
            loss = criterion(pred, tgt) / ACCUM_STEPS

        if not torch.isfinite(loss):
            torch.save(b, "nan_batch.pt")
            raise RuntimeError(f"Non-finite loss at epoch {epoch} step {step}")

        scaler.scale(loss).backward()

        with torch.no_grad():
            p_f = pred.detach().float()
            total_loss += loss.item() * ACCUM_STEPS
            total_mae  += l1_loss(torch.expm1(p_f), b.y.float().squeeze(-1)).item()
            n_step     += 1

        if (step + 1) % ACCUM_STEPS == 0 or (step + 1) == len(train_loader):
            scaler.unscale_(optim)
            grad_norm = nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            if torch.isfinite(torch.as_tensor(grad_norm)):
                scaler.step(optim)
            scaler.update()
            optim.zero_grad(set_to_none=True)

        bar.set_postfix({"loss": f"{total_loss/n_step:.3f}", "mae": f"{total_mae/n_step:.3f}"})

    full_ds.eval()
    scheduler.step()
    val_mae, val_rmse = evaluate(val_loader, "  val")
    torch.cuda.empty_cache()

    history["loss"].append(total_loss / max(n_step, 1))
    history["train_mae"].append(total_mae / max(n_step, 1))
    history["val_mae"].append(val_mae)
    history["val_rmse"].append(val_rmse)

    star = ""
    if val_mae < best_val_mae:
        best_val_mae = val_mae
        torch.save({
            "epoch"     : epoch,
            "model"     : model.state_dict(),
            "optim"     : optim.state_dict(),
            "scheduler" : scheduler.state_dict(),
            "scaler"    : scaler.state_dict(),
            "val_mae"   : val_mae,
            "val_rmse"  : val_rmse,
            "history"   : history,
            "config"    : {
                "model": MODEL, "node_dim": NODE_DIM, "edge_dim": EDGE_DIM,
                "num_layers": NUM_LAYERS, "max_k": MAX_K,
            },
        }, ckpt_path)
        star = "  ★"

    t = time.perf_counter() - t0
    print(
        f"  loss={history['loss'][-1]:.4f}  "
        f"MAE={val_mae:.4f} eV  RMSE={val_rmse:.4f} eV  "
        f"lr={scheduler.get_last_lr()[0]:.1e}  {t:.0f}s{star}"
    )

print(f"\nDone.  Best val band-gap MAE = {best_val_mae:.4f} eV")
print(f"Checkpoint: {ckpt_path}  ← download before session ends!")


## 9 · Test evaluation

In [ ]:
ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
model.load_state_dict(ckpt["model"])
print(f"Loaded epoch {ckpt['epoch']}  val MAE = {ckpt['val_mae']:.4f} eV")

test_mae, test_rmse = evaluate(test_loader, "test")
print(f"\nTest Band-Gap  MAE  : {test_mae:.4f} eV")
print(f"Test Band-Gap  RMSE : {test_rmse:.4f} eV")


## 10 · Training curves

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle(f"{Model.__name__}  max_k={MAX_K}  layers={NUM_LAYERS}  dim={NODE_DIM}", fontsize=12)

axes[0].plot(history["loss"],      label="Huber loss", color="tab:blue")
axes[0].plot(history["train_mae"], label="Train MAE",  color="tab:green", linestyle="--")
axes[0].set(title="Training", xlabel="Epoch", ylabel="Value")
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(history["val_mae"],  label="Val MAE",  color="tab:orange")
axes[1].plot(history["val_rmse"], label="Val RMSE", color="tab:red", linestyle="--")
axes[1].axhline(test_mae,  color="tab:orange", linestyle=":", alpha=0.8,
                label=f"Test MAE  {test_mae:.3f} eV")
axes[1].axhline(test_rmse, color="tab:red",    linestyle=":", alpha=0.8,
                label=f"Test RMSE {test_rmse:.3f} eV")
axes[1].set(title="Band Gap (eV)", xlabel="Epoch", ylabel="eV")
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig("/content/training_curves.png", dpi=150)
plt.show()
print("Saved → /content/training_curves.png")


## 11 · Download checkpoint before session ends

In [ ]:
from google.colab import files
import os

if os.path.exists(ckpt_path):
    print(f"Downloading {CKPT_NAME} ({os.path.getsize(ckpt_path)/1e6:.1f} MB) …")
    files.download(ckpt_path)
else:
    print("No checkpoint found — has training completed?")